HANDLING IMBALANCED DATASETS IN MACHINE LEARNING

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
df=pd.read_csv("c:\\Users\Omojire\Downloads\\customer_churn.csv")


<>:3: SyntaxWarning: invalid escape sequence '\O'
<>:3: SyntaxWarning: invalid escape sequence '\O'
C:\Users\Omojire\AppData\Local\Temp\ipykernel_16984\723855228.py:3: SyntaxWarning: invalid escape sequence '\O'
  df=pd.read_csv("c:\\Users\Omojire\Downloads\\customer_churn.csv")


In [2]:
df=df[df['TotalCharges']!=' ']
df['TotalCharges']=pd.to_numeric(df.TotalCharges)# coerce converts all objects to float64 regardless of their data type
df=df.drop(columns=df[['customerID']],axis='columns')


In [3]:
df.replace('No phone service','No',inplace=True)
df.replace('No internet service','No',inplace=True)
df['Churn']=df['Churn'].apply(lambda x :1 if x=='Yes' else 0)


In [4]:
dummies=pd.get_dummies(df[['DeviceProtection','Partner','TechSupport' ,'StreamingTV' ,'StreamingMovies','Contract','MultipleLines',
                     'InternetService','OnlineSecurity', 'PaymentMethod','OnlineBackup','gender','Dependents', 'PhoneService','PaperlessBilling']],drop_first=True)
df=pd.concat([df,dummies],axis=1)
df=df.drop(columns=df[['Partner','DeviceProtection','TechSupport' ,'StreamingTV' ,'StreamingMovies','Contract','MultipleLines',
                     'InternetService','PaymentMethod','OnlineSecurity','OnlineBackup','gender','Dependents','PhoneService','PaperlessBilling']],axis='columns')


In [5]:
features=df.drop(columns=df[['Churn']],axis='columns')
#features=features.astype(int)# so tensorflow can read 0 and 1 instead of True and False
target=df['Churn']
print(target.value_counts())
'''we can see that the dataset is imbalanced,the model will learn to predict the class 0 easily because it is the majority class 
and this is a very big issue ,
I will go ahead to train an Artificial neural network on it ,then retrain the model after
applying different techniques for handling imbalanced datasets and compare the improvements to 
the normal artificial neural network''' 

Churn
0    5163
1    1869
Name: count, dtype: int64


'we can see that the dataset is imbalanced,the model will learn to predict the class 0 easily because it is the majority class \nand this is a very big issue ,\nI will go ahead to train an Artificial neural network on it ,then retrain the model after\napplying different techniques for handling imbalanced datasets and compare the improvements to \nthe normal artificial neural network'

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(features,target,test_size=0.2,random_state=42)
X_train['TotalCharges']=X_train['TotalCharges']/X_train['TotalCharges'].max()
X_train['MonthlyCharges']=X_train['MonthlyCharges']/X_train['MonthlyCharges'].max()
X_test['TotalCharges']=X_test['TotalCharges']/X_test['TotalCharges'].max()
X_test['MonthlyCharges']=X_test['MonthlyCharges']/X_test['MonthlyCharges'].max()


In [7]:
import tensorflow as tf 
from tensorflow import keras 

model=keras.Sequential([keras.layers.Dense(26,input_shape=(23,),activation='relu'),
                         keras.layers.Dropout(0.5),
                        keras.layers.Dense(15,activation='relu'),
                        keras.layers.Dropout(0.5),
                        keras.layers.Dense(1,activation='sigmoid')])
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.fit(X_train,y_train,epochs=20)

print(model.evaluate(X_test,y_test))

predicted = model.predict(X_test)


C:\Users\Omojire\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6276 - loss: 1.3351
Epoch 2/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6811 - loss: 0.7061
Epoch 3/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7152 - loss: 0.5946
Epoch 4/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7250 - loss: 0.5603
Epoch 5/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7291 - loss: 0.5485
Epoch 6/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7323 - loss: 0.5315
Epoch 7/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7321 - loss: 0.5179
Epoch 8/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7332 - loss: 0.5071
Epoch 9/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7335 - loss: 0.5013
Epoch 10/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7410 - loss: 0.4971
Epoch 11/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7557 - loss: 0.4848
Epoch 12/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

In [8]:
y_pred=[]
for i in range(len(predicted)):
    if predicted[i]<0.5:
        y_pred.append(0)
    else:
        y_pred.append(1)

print(y_pred)


[0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [9]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

'''The imbalance has affected the result as seen in the low F-1 score of class 1,we will go ahead to apply those techniques 
for tackling imbalanced datasets one by one and compare the improvements to this result'''

              precision    recall  f1-score   support

           0       0.81      0.93      0.86      1033
           1       0.66      0.39      0.49       374

    accuracy                           0.78      1407
   macro avg       0.73      0.66      0.67      1407
weighted avg       0.77      0.78      0.76      1407



'The imbalance has affected the result as seen in the low F-1 score of class 1,we will go ahead to apply those techniques \nfor tackling imbalanced datasets one by one and compare the improvements to this result'

In [10]:
'''Let's create a function so that we can train our model without repeating the same thing everytime'''
def ANN(model,x,y,epochs):
    
    model.fit(X_train,y_train,epochs=epochs)
    print(model.evaluate(X_test,y_test))
    predicted = model.predict(X_test)

    y_pred=[]
    for i in range(len(predicted)):
      if predicted[i]<0.5:
         y_pred.append(0)
      else:
         y_pred.append(1)

    
    from sklearn.metrics import classification_report
    print(classification_report(y_test,y_pred))


METHOD 1:UNDERSAMPLING THE MAJORITY CLASS

In [11]:

df_class_0=df[df['Churn']==0]
df_class_1=df[df['Churn']==1]


df_class_0.shape,df_class_1.shape

((5163, 24), (1869, 24))

In [12]:
df_count_class_0,df_count_class_1=df.Churn.value_counts()

In [13]:
df_count_class_0,df_count_class_1

(5163, 1869)

In [14]:
df_class_0_under=df_class_0.sample(df_count_class_1)#randomly selects 1869 samples in class 0
df_class_0.shape
df_class_0_under.shape,df_class_1.shape
df_class_0_under.Churn.value_counts() ,df_class_1.Churn.value_counts()

(Churn
 0    1869
 Name: count, dtype: int64,
 Churn
 1    1869
 Name: count, dtype: int64)

In [15]:
df_final=pd.concat([df_class_0_under,df_class_1],axis=0)
y=df_final['Churn']
x=df_final.drop(columns=['Churn'])
x.shape,  y.value_counts()

((3738, 23),
 Churn
 0    1869
 1    1869
 Name: count, dtype: int64)

In [16]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=15,stratify=y)
y_train.value_counts(), y_test.value_counts()

(Churn
 0    1495
 1    1495
 Name: count, dtype: int64,
 Churn
 1    374
 0    374
 Name: count, dtype: int64)

In [17]:
ANN(model,x,y,200)
'''we can see that the F-1 score of both classes are closer compared to the metrics in the pre-trained,and now both classes are balanced
The overall accuracy (~74%) might look lower than before, but now the model is fair to both classes instead of just predicting “0” all the time.
That’s what matters when your real goal is to detect churners'''


Epoch 1/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5184 - loss: 57.1517 
Epoch 2/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5060 - loss: 4.9639
Epoch 3/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5067 - loss: 1.7288
Epoch 4/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5395 - loss: 0.8869
Epoch 5/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5405 - loss: 0.7165
Epoch 6/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5411 - loss: 0.7060
Epoch 7/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5712 - loss: 0.6940
Epoch 8/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5535 - loss: 0.7075
Epoch 9/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5676 - loss: 0.6822
Epoch 10/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5592 - loss: 0.6987
Epoch 11/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5622 - loss: 0.7032
Epoch 12/200
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accurac

'we can see that the F-1 score of both classes are closer compared to the metrics in the pre-trained,and now both classes are balanced\nThe overall accuracy (~72%) might look lower than before, but now the model is fair to both classes instead of just predicting “0” all the time.\nThat’s what matters when your real goal is to detect churners'

METHOD 2:OVERSAMPLING THE MINORITY CLASS

In [18]:

df_class_0=df[df['Churn']==0]
df_class_1=df[df['Churn']==1]


df_count_class_0,df_count_class_1=df.Churn.value_counts()
df_count_class_0,df_count_class_1

(5163, 1869)

In [19]:
df_class_one=df_class_1.sample(df_count_class_0,replace=True)
df_class_one.Churn.unique

<bound method Series.unique of 1639    1
2589    1
3600    1
1304    1
3544    1
       ..
3006    1
1701    1
6620    1
5383    1
2619    1
Name: Churn, Length: 5163, dtype: int64>

In [20]:
df_class_0.shape,df_class_one.shape

((5163, 24), (5163, 24))

In [21]:
df_final=pd.concat([df_class_0,df_class_one],axis=0)
y=df_final['Churn']
x=df_final.drop(columns=['Churn'])
x.shape,  y.value_counts()

((10326, 23),
 Churn
 0    5163
 1    5163
 Name: count, dtype: int64)

In [22]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=15,stratify=y)
y_train.value_counts(), y_test.value_counts()

(Churn
 1    4130
 0    4130
 Name: count, dtype: int64,
 Churn
 1    1033
 0    1033
 Name: count, dtype: int64)

In [23]:
ANN(model,x,y,100)
'''The metrics here are also better compared to the pre-trained model'''

Epoch 1/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7000 - loss: 0.5789
Epoch 2/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7025 - loss: 0.5801
Epoch 3/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6886 - loss: 0.5860
Epoch 4/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7071 - loss: 0.5725
Epoch 5/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6880 - loss: 0.5855
Epoch 6/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6904 - loss: 0.5823
Epoch 7/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6789 - loss: 0.5947
Epoch 8/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7104 - loss: 0.5766
Epoch 9/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6978 - loss: 0.5778
Epoch 10/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6953 - loss: 0.5859
Epoch 11/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6906 - loss: 0.5821
Epoch 12/100
259/259 ━━━━━━━━━━━━━━━━━━━━

'The metrics here are also better compared to the pre-trained model'

METHOD 3:SMOTE(USES K-NEAREST NEIGHBOURS ALGORITHM)

In [24]:
x=df.drop(columns=['Churn'],axis='columns')
y=df['Churn']
y.value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

pip install imbalanced-learn

In [25]:
from imblearn.over_sampling import SMOTE

smote=SMOTE(sampling_strategy='minority')
X_sm,y_sm=smote.fit_resample(x,y)

y_sm.value_counts()


Churn
0    5163
1    5163
Name: count, dtype: int64

In [26]:
X_train,X_test,y_train,y_test=train_test_split(X_sm,y_sm,test_size=0.2,random_state=15,stratify=y_sm)
y_train.value_counts(),  y_test.value_counts()

(Churn
 1    4130
 0    4130
 Name: count, dtype: int64,
 Churn
 1    1033
 0    1033
 Name: count, dtype: int64)

In [27]:
ANN(model,X_sm,y_sm,100)


Epoch 1/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6984 - loss: 0.5772
Epoch 2/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7120 - loss: 0.5706
Epoch 3/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7024 - loss: 0.5741
Epoch 4/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7058 - loss: 0.5706
Epoch 5/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6996 - loss: 0.5742
Epoch 6/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7000 - loss: 0.5733
Epoch 7/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6960 - loss: 0.5739
Epoch 8/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7023 - loss: 0.5711
Epoch 9/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7100 - loss: 0.5655
Epoch 10/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7103 - loss: 0.5689
Epoch 11/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7071 - loss: 0.5713
Epoch 12/100
259/259 ━━━━━━━━━━━━━━━━━━━━

METHOD 4:ENSEMBLE METHODS

In [28]:
df.Churn.value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [29]:
x=df.drop(columns=['Churn'],axis='columns')
y=df['Churn']


In [30]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=15,stratify=y)

In [31]:
y_train.value_counts()

Churn
0    4130
1    1495
Name: count, dtype: int64

In [32]:
4130/3

1376.6666666666667

In [33]:
df1=df.copy()
df1['Churn']=y_train
df1_class_0=df1[df1.Churn==0]
df1_class_1=df1[df1.Churn==1]

In [34]:
df1_class_0.shape,df1_class_1.shape

((4130, 24), (1495, 24))

In [35]:
def get_train_batch(df_majority,df_minority,start,end):
    df_train=pd.concat([df_majority[start:end],df_minority],axis=0)

    X_train=df_train.drop('Churn',axis='columns')
    y_train=df_train['Churn']
    
    return X_train,y_train
    

In [43]:
X_train,y_train=get_train_batch(df1_class_0,df1_class_1,0,1495)
y_pred_1=ANN(model,X_train,y_train,100)
y_pred_1=y_pred

Epoch 1/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7047 - loss: 0.5754
Epoch 2/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7207 - loss: 0.5562
Epoch 3/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7097 - loss: 0.5642
Epoch 4/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6997 - loss: 0.5712
Epoch 5/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7070 - loss: 0.5625
Epoch 6/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7060 - loss: 0.5627
Epoch 7/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7107 - loss: 0.5636
Epoch 8/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7040 - loss: 0.5759
Epoch 9/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7074 - loss: 0.5673
Epoch 10/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7107 - loss: 0.5748
Epoch 11/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7084 - loss: 0.5671
Epoch 12/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy:

In [44]:
X_train,y_train=get_train_batch(df1_class_0,df1_class_1,1495,2990)
y_pred_2=ANN(model,X_train,y_train,100)
y_pred_2=y_pred

Epoch 1/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7227 - loss: 0.5511
Epoch 2/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7084 - loss: 0.5651
Epoch 3/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7187 - loss: 0.5574
Epoch 4/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7224 - loss: 0.5568
Epoch 5/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7371 - loss: 0.5453
Epoch 6/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7104 - loss: 0.5673
Epoch 7/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7087 - loss: 0.5701
Epoch 8/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7184 - loss: 0.5537
Epoch 9/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7094 - loss: 0.5694
Epoch 10/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7074 - loss: 0.5698
Epoch 11/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7177 - loss: 0.5651
Epoch 12/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy:

In [45]:
X_train,y_train=get_train_batch(df1_class_0,df1_class_1,2990,4130)
y_pred_3=ANN(model,X_train,y_train,100)
y_pred_3=y_pred

Epoch 1/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7446 - loss: 0.5498
Epoch 2/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7324 - loss: 0.5570
Epoch 3/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7332 - loss: 0.5573
Epoch 4/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7408 - loss: 0.5542
Epoch 5/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7332 - loss: 0.5561
Epoch 6/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7408 - loss: 0.5504
Epoch 7/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7427 - loss: 0.5453
Epoch 8/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7374 - loss: 0.5536
Epoch 9/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7366 - loss: 0.5486
Epoch 10/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7378 - loss: 0.5500
Epoch 11/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7378 - loss: 0.5456
Epoch 12/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy:

In [46]:
'''we would take the average of the 3 predictions(majority vote)'''
y_pred_final=[]
for i in range(len(y_pred)):
    ones=y_pred_1[i]+y_pred_2[i]+y_pred_3[i]
    if ones>1:
        y_pred_final.append(1)
    else:
        y_pred_final.append(0)
    

In [47]:
print(classification_report(y_test,y_pred_final))

              precision    recall  f1-score   support

           0       0.73      0.84      0.78      1033
           1       0.24      0.14      0.18       374

    accuracy                           0.65      1407
   macro avg       0.49      0.49      0.48      1407
weighted avg       0.60      0.65      0.62      1407



In [ ]:
'''Ensemble methods didn't increase the performance of the model for this data,but this doesn't mean it is a bad method to tackle 
unbalanced datasets in machine learning.
This is machine learning,it's about trying,tuning and failing till you get the desired results'''